# Test 1 — Klasse 1: Satzebene-Erkennung über semantische Nearest-Neighbor-Suche

Prüft den Klasse-1-Ansatz aus [konzept_hybrid_technik_erkennung.md](../../konzept_hybrid_technik_erkennung.md#klasse-1--satzebene-nn-only-kein-llm):
Embedding eines einzelnen Satzes, NN-Suche gegen verifizierte Technik-Beispiele aus der
Pipeline-Datenbank, Label wird bei ausreichender Nähe übernommen — **kein LLM-Call**.

**Offene Punkte aus dem Konzept, die dieser Test adressiert:**
- Punkt 2: Distanz-Schwellenwert für Klasse 1 kalibrieren
- Punkt 4: Taugt `paraphrase-multilingual-mpnet-base-v2` besser als der Chroma-Default
  (`all-MiniLM-L6-v2`), um strukturelle/rhetorische statt rein thematische Nähe abzubilden?

**Isolation:** Diese Notebook liest nur lesend aus der laufenden Pipeline-DB (Technik-Zitate
als Trainingsmaterial) und schreibt in eine eigene, lokale ChromaDB
(`./chroma_nn_store/`, nicht Teil des Repos). Die Produktionsdaten werden nicht verändert.


## 1 — Setup

In [ ]:
import json
from pathlib import Path

import sys
sys.path.insert(0, "../../../../src")

import chromadb
from chromadb.utils import embedding_functions

from news_analyser.repositories.chroma_client import get_client as get_pipeline_client

# Kandidat aus dem Konzept (Zeile 116-118) -- gegen den Chroma-Default all-MiniLM-L6-v2 testen,
# indem EMBEDDING_MODEL hier einfach umgeschaltet wird.
EMBEDDING_MODEL = "paraphrase-multilingual-mpnet-base-v2"

NN_STORE_PATH = Path("./chroma_nn_store")
NN_COLLECTION = "technique_examples_klasse1"

embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL)
nn_client = chromadb.PersistentClient(path=str(NN_STORE_PATH))

print(f"Embedding-Modell: {EMBEDDING_MODEL}")
print(f"NN-Store: {NN_STORE_PATH.resolve()}")


## 2 — Technik-Zitate aus der Pipeline-Datenbank laden

Liest alle Artikel-Metadaten aus der laufenden `articles`-Collection (siehe
`src/news_analyser/repositories/db_storage.py`) und extrahiert aus dem darin abgelegten
`analysis_json` jede einzelne `detected_techniques`-Instanz (`technique` + `quote`).
Das sind die bereits vom Zwei-Pass-LLM verifizierten Beispiele, die als Referenzmenge für die
NN-Suche dienen.


In [ ]:
pipeline_col = get_pipeline_client().get_collection("articles")
all_docs = pipeline_col.get(include=["metadatas"])

examples = []  # {technique, quote, source_url}
for meta in all_docs["metadatas"]:
    analysis = json.loads(meta.get("analysis_json", "{}"))
    url = analysis.get("source_url", "")
    for t in analysis.get("detected_techniques", []):
        quote = t.get("quote", "").strip()
        technique = t.get("technique", "").strip()
        if quote and technique:
            examples.append({"technique": technique, "quote": quote, "source_url": url})

print(f"{len(examples)} Technik-Zitate aus {len(all_docs['metadatas'])} Artikeln geladen")

from collections import Counter
Counter(e["technique"] for e in examples).most_common()


## 3 — NN-Store befüllen

Eigene lokale Collection, embedded mit dem oben gewählten `EMBEDDING_MODEL` (nicht dem Projekt-Default).

In [ ]:
nn_col = nn_client.get_or_create_collection(
    name=NN_COLLECTION,
    embedding_function=embed_fn,
    metadata={"hnsw:space": "cosine"},
)

nn_col.upsert(
    ids=[f"{i:04d}_{e['technique'][:20]}" for i, e in enumerate(examples)],
    documents=[e["quote"] for e in examples],
    metadatas=[{"technique": e["technique"], "source_url": e["source_url"]} for e in examples],
)

print(f"NN-Store befuellt: {nn_col.count()} Eintraege")


## 4 — Testsätze gegen den NN-Store vergleichen

Neue Sätze, satzweise gegen den NN-Store.
`expected_technique` ist die manuell erwartete Technik (`None` = Negativbeispiel, soll nichts
matchen).

**Stufe 1 — Self-Recognition (unten eingetragen):** Sätze stammen aus demselben Artikel, dessen
Zitate gerade den Store befüllt haben. Erwartung: Distanz ≈ 0, nahezu perfekter Match. Zeigt nur,
dass die Mechanik (Embedding, Query, Store) funktioniert — sagt nichts über Generalisierung aus.

**Stufe 2 — Generalisierung (noch zu ergänzen):** Sätze aus einem Artikel, der *nicht* im
NN-Store steckt, oder umformulierte Varianten der Stufe-1-Sätze. Erst das prüft, ob die
NN-Suche eine Technik auch in unbekanntem Text wiedererkennt und liefert den echten
Distanz-Schwellenwert für Konzept-Punkt 2.

In [ ]:
# Stufe 1 — Self-Recognition: aus dem letzten Artikel in der Pipeline-DB
# (Focus, "Trump: Waffenruhe mit Iran vorbei", 2026-07-08) -- dieselben Zitate stecken
# bereits im NN-Store, siehe Hinweis oben.
test_sentences = [
    {"text": "Trump: Waffenruhe mit Iran vorbei, „die sind krank”", "expected_technique": "Framing"},
    {"text": "Trump schimpfte über die Mullahs: „Die sind krank. Mit denen stimmt etwas nicht.“", "expected_technique": "Loaded Language"},
    {"text": "Die USA greifen Angaben aus Teheran zufolge nach ihren nächtlichen Bombardierungen weiter an der iranischen Südküste an.", "expected_technique": "Omission"},
    {"text": "Nahe der Hafenstadt Buschehr seien mindestens zwei Projektile eingeschlagen, berichtete die iranische Nachrichtenagentur Tasnim am Vormittag.", "expected_technique": None},
    {"text": "Der Iran müsse aber jetzt wirklich verstehen, dass es Zeit für ernsthafte Verhandlungen sei, sagte der CDU-Politiker dem Sender NDR Info.", "expected_technique": None},
]


In [ ]:
K = 3

rows = []
for case in test_sentences:
    hits = nn_col.query(
        query_texts=[case["text"]],
        n_results=min(K, nn_col.count()),
        include=["metadatas", "documents", "distances"],
    )
    for rank in range(len(hits["ids"][0])):
        rows.append({
            "satz":             case["text"][:60],
            "erwartet":         case["expected_technique"],
            "rang":             rank + 1,
            "gefunden_technik": hits["metadatas"][0][rank]["technique"],
            "distanz":          round(hits["distances"][0][rank], 3),
            "zitat_match":      hits["documents"][0][rank][:80],
        })

import pandas as pd
df = pd.DataFrame(rows)
df


## 5 — Ergebnisse dokumentieren

Notebook-Outputs werden vor dem Commit von `nbstripout` entfernt — Fazit, kalibrierter
Schwellenwert und die Embedding-Modell-Entscheidung deshalb in
[results.md](results.md) in diesem Ordner festhalten (Vorlage bereits angelegt).